In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.utils import to_categorical

#load in CIFAR-10
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

In [ ]:
#step 2 - Resize for MobileNetV2 (224x224)
from tensorflow.image import resize

X_train = tf.image.resize(X_train, [224, 224])
X_test = tf.image.resize(X_test, [224, 224])

In [ ]:
#step 3 - Load MobileNetV2 without top classification head
from tensorflow.keras.applications import MobileNetV2

#load MobileNetV2 base model
base_model = MobileNetV2(weights='imagenet',
                         include_top=False, #without top classification head
                         input_shape=(224, 224, 3))

base_model.trainable = False  #going with freezed pretrained layers (for now)

In [ ]:
from akida_models import fetch_file
from quantizeml import load_model

model = load_model(base_model)

samples = X_train[:1024].astype(np.float32)

In [ ]:
from quantizeml.models import QuantizationParams
qparams4 = QuantizationParams(input_weight_bits=8, weight_bits=4, activation_bits=4)

from quantizeml.models import quantize
quantized_model = quantize(model, qparams=qparams4, samples=samples)

from cnn2snn import convert
akida_model = convert(quantized_model)

In [ ]:
import numpy as np
import akida

# 1. Did quantization keep the accuracy? Compare float and quantized Keras on 2000 test images.
_, (x_test, y_test) = keras.datasets.cifar10.load_data()
x, y = x_test[:2000].astype(np.float32), y_test[:2000].ravel()
print("float    :", np.mean(model.predict(x, verbose=0).argmax(axis=1) == y))
print("quantized:", np.mean(quantized_model.predict(x, verbose=0).argmax(axis=1) == y))

# 2. Did conversion produce a sensible Akida model?
akida_model.summary()
print("input shape:", akida_model.input_shape)

# 3. Does the Akida model agree with the quantized Keras model? (simulator, no card)
out = akida_model.forward(x_test[:2000].astype(np.uint8))
akida_pred = np.asarray(out).reshape(2000, -1).argmax(axis=1)
quant_pred = quantized_model.predict(x, verbose=0).argmax(axis=1)
print("akida accuracy        :", np.mean(akida_pred == y))
print("akida vs quantized agreement:", np.mean(akida_pred == quant_pred))

# 4. Is the card visible? (server only)
print("devices:", akida.devices())